# Talk2Mind — Train Text, Audio & Video Models (Google Colab)

Trains the 3 models your FastAPI app expects:
```
text_model_final.h5
audio_model_final.h5
video_model_final.h5
text_vectorizer_vocab.pkl
```

**Datasets used:**
- **Dataset 1 — CMU-MOSEI** (`Talk2Mind_data/CMU-MOSEI`): transcripts + audio chunks + emotion labels -> trains the **Text model**, and contributes to the **Audio model**.
- **Dataset 2 — Multimodel_Dataset** (`Talk2Mind_data/Multimodel_Dataset`): TESS-style audio (`Audio_Dataset/`, emotion in filename) + RAVDESS video (`Video_Dataset/`, emotion coded in filename) -> also contributes to the **Audio model**, and is the sole source for the **Video model**. (EEG data in this dataset is not used — no EEG model was requested.)

Run cells top to bottom. Read the diagnostic print-outs after each loading step before committing to a long training run.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Dataset paths (as you provided) + sanity check

In [ ]:
DATASET1_DIR = "/content/drive/MyDrive/Talk2Mind_data/CMU-MOSEI"
DATASET2_DIR = "/content/drive/MyDrive/Talk2Mind_data/Multimodel_Dataset"

# Where trained models + vocab get saved (also on Drive, so they survive disconnects).
SAVE_DIR = "/content/drive/MyDrive/Talk2Mind_data/models"

import os
os.makedirs(SAVE_DIR, exist_ok=True)

for p in [DATASET1_DIR, DATASET2_DIR]:
    assert os.path.isdir(p), f"Not found: {p}  (check your Drive folder name/path)"
    print(p, "->", os.listdir(p))


## 3. Install extra dependencies

TensorFlow + OpenCV already ship with Colab — we only need librosa/soundfile for audio features.

In [ ]:
!pip install -q librosa==0.10.2.post1 soundfile==0.12.1


## 4. Global config

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
import librosa
import cv2
import pickle
import re
from sklearn.model_selection import train_test_split

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

# ---- shared label space across ALL three models ----
EMOTION_LABELS = ["happy", "sad", "angry", "surprise", "disgust", "fear"]
NUM_CLASSES = len(EMOTION_LABELS)

# ---- audio feature settings ----
AUDIO_SAMPLE_RATE = 16000
AUDIO_DURATION_SEC = 4.0
AUDIO_N_MFCC = 40
AUDIO_N_MELS = 64
AUDIO_HOP_LENGTH = 512
AUDIO_N_FFT = 1024
AUDIO_TIME_STEPS = int(AUDIO_SAMPLE_RATE * AUDIO_DURATION_SEC / AUDIO_HOP_LENGTH) + 1
AUDIO_FEATURE_DIM = AUDIO_N_MFCC + AUDIO_N_MELS + 2  # + rms + f0

# ---- video feature settings ----
VIDEO_NUM_FRAMES = 16
VIDEO_FRAME_SIZE = 96

# ---- text settings ----
TEXT_VOCAB_SIZE = 20000
TEXT_MAX_LEN = 50
TEXT_EMBED_DIM = 128

# ---- training settings (EDIT THESE to trade off speed vs. accuracy) ----
BATCH_SIZE = 8
EPOCHS_TEXT = 15
EPOCHS_AUDIO = 15
EPOCHS_VIDEO = 10
LEARNING_RATE = 1e-4


## 5. Feature extraction helpers (audio + video)

In [ ]:
def load_and_fix_length(path, sr=AUDIO_SAMPLE_RATE, duration=AUDIO_DURATION_SEC):
    y, _ = librosa.load(path, sr=sr, mono=True)
    target_len = int(sr * duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)), mode="constant")
    else:
        y = y[:target_len]
    return y


def extract_audio_features(path):
    """MFCC + log-mel + RMS energy + pitch (f0), fixed-length, z-normalized."""
    path = path.numpy().decode("utf-8") if hasattr(path, "numpy") else path
    y = load_and_fix_length(path)

    mfcc = librosa.feature.mfcc(y=y, sr=AUDIO_SAMPLE_RATE, n_mfcc=AUDIO_N_MFCC,
                                 n_fft=AUDIO_N_FFT, hop_length=AUDIO_HOP_LENGTH)
    mel = librosa.feature.melspectrogram(y=y, sr=AUDIO_SAMPLE_RATE, n_mels=AUDIO_N_MELS,
                                          n_fft=AUDIO_N_FFT, hop_length=AUDIO_HOP_LENGTH)
    log_mel = librosa.power_to_db(mel)
    rms = librosa.feature.rms(y=y, hop_length=AUDIO_HOP_LENGTH)
    try:
        f0, _, _ = librosa.pyin(y, fmin=librosa.note_to_hz("C2"), fmax=librosa.note_to_hz("C7"),
                                 sr=AUDIO_SAMPLE_RATE, hop_length=AUDIO_HOP_LENGTH)
        f0 = np.nan_to_num(f0)[np.newaxis, :]
    except Exception:
        f0 = np.zeros_like(rms)

    T = min(mfcc.shape[1], log_mel.shape[1], rms.shape[1], f0.shape[1])
    feat = np.concatenate([mfcc[:, :T], log_mel[:, :T], rms[:, :T], f0[:, :T]], axis=0).T

    if feat.shape[0] < AUDIO_TIME_STEPS:
        feat = np.concatenate([feat, np.zeros((AUDIO_TIME_STEPS - feat.shape[0], feat.shape[1]))], axis=0)
    else:
        feat = feat[:AUDIO_TIME_STEPS, :]

    mean, std = feat.mean(), feat.std() + 1e-6
    return ((feat - mean) / std).astype(np.float32)


_face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

def _crop_largest_face(frame_bgr):
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    faces = _face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
    if len(faces) == 0:
        return frame_bgr
    x, y, w, h = max(faces, key=lambda box: box[2] * box[3])
    return frame_bgr[y:y + h, x:x + w]


def extract_face_sequence(video_path, num_frames=VIDEO_NUM_FRAMES, frame_size=VIDEO_FRAME_SIZE):
    path = video_path.numpy().decode("utf-8") if hasattr(video_path, "numpy") else video_path
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = set(np.round(np.linspace(0, max(total - 1, 0), num_frames)).astype(int).tolist())

    grabbed, frame_i = {}, 0
    while cap.isOpened() and len(grabbed) < len(indices):
        ret, frame = cap.read()
        if not ret:
            break
        if frame_i in indices:
            face = _crop_largest_face(frame)
            face = cv2.resize(face, (frame_size, frame_size))
            face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
            grabbed[frame_i] = face.astype(np.float32) / 255.0
        frame_i += 1
    cap.release()

    ordered = [grabbed[i] for i in sorted(grabbed.keys())]
    if len(ordered) == 0:
        ordered = [np.zeros((frame_size, frame_size, 3), dtype=np.float32)]
    while len(ordered) < num_frames:
        ordered.append(ordered[-1])
    return np.stack(ordered[:num_frames], axis=0)

print("Feature extraction helpers ready.")


## 6. Load Dataset 1 — CMU-MOSEI (text + audio + emotion labels)

Auto-detects your CSV column names. If detection fails, it prints all columns
found so you can fill in `MANUAL_COLUMNS` below and re-run.

In [ ]:
LABELS_DIR = os.path.join(DATASET1_DIR, "Labels")

SPLIT_CSV = {
    "train": os.path.join(LABELS_DIR, "Data_Train_modified.csv"),
    "val":   os.path.join(LABELS_DIR, "Data_Val_modified.csv"),
    "test":  os.path.join(LABELS_DIR, "Data_Test_modified.csv"),
}
SPLIT_AUDIO_DIR = {
    "train": os.path.join(DATASET1_DIR, "Audio_chunk", "Train_modified"),
    "val":   os.path.join(DATASET1_DIR, "Audio_chunk", "Val_modified"),
    "test":  os.path.join(DATASET1_DIR, "Audio_chunk", "Test_modified"),
}

# If auto-detection below fails for your actual CSV, hardcode the real column
# names here instead, e.g. {"video_id": "clip_id", "start": "start_time", ...}
MANUAL_COLUMNS = {}


def detect_column(df, candidates):
    for c in candidates:
        for col in df.columns:
            if col.strip().lower() == c:
                return col
    return None


def resolve_columns(df):
    cols = {}
    cols["video_id"] = MANUAL_COLUMNS.get("video_id") or detect_column(df, ["video_id", "videoid", "video", "vid"])
    cols["start"] = MANUAL_COLUMNS.get("start") or detect_column(df, ["start", "start_time", "segment_start"])
    cols["end"] = MANUAL_COLUMNS.get("end") or detect_column(df, ["end", "end_time", "segment_end"])
    cols["text"] = MANUAL_COLUMNS.get("text") or detect_column(df, ["text", "transcript", "sentence"])
    for emo in EMOTION_LABELS:
        cols[emo] = MANUAL_COLUMNS.get(emo) or detect_column(df, [emo])
    missing = [k for k, v in cols.items() if v is None]
    if missing:
        raise ValueError(
            f"Could not auto-detect columns {missing}. "
            f"Actual columns in CSV: {list(df.columns)}. "
            f"Fill in MANUAL_COLUMNS above with the real names and re-run this cell."
        )
    return cols


def load_cmu_mosei_split(split):
    csv_path = SPLIT_CSV[split]
    if not os.path.exists(csv_path):
        print(f"[CMU-MOSEI] no CSV for split '{split}' at {csv_path} — skipping.")
        return pd.DataFrame(columns=["audio_path", "text"] + EMOTION_LABELS)

    df = pd.read_csv(csv_path)
    cols = resolve_columns(df)

    audio_dir = SPLIT_AUDIO_DIR[split]
    rows = []
    for _, row in df.iterrows():
        vid = row[cols["video_id"]]
        start = float(row[cols["start"]])
        end = float(row[cols["end"]])
        fname = f"{vid}_{start:.4f}_{end:.4f}.wav"
        path = os.path.join(audio_dir, fname)
        if os.path.exists(path):
            rec = {"audio_path": path, "text": str(row[cols["text"]])}
            for emo in EMOTION_LABELS:
                rec[emo] = float(row[cols[emo]])
            rows.append(rec)

    out = pd.DataFrame(rows)
    print(f"[CMU-MOSEI] split='{split}': {len(out)}/{len(df)} rows matched to an audio file.")
    return out


cmu_train_df = load_cmu_mosei_split("train")
cmu_val_df = load_cmu_mosei_split("val")
cmu_test_df = load_cmu_mosei_split("test")

cmu_train_df.head()


## 7. Load Dataset 2 — Audio (TESS-style, filename-labeled) and Video (RAVDESS, filename-labeled)

In [ ]:
# ---- Dataset 2 audio: emotion is the last "_"-separated token in the filename ----
EMOTION_MAP_DS2_AUDIO = {
    "angry": "angry", "disgust": "disgust", "fear": "fear",
    "happy": "happy", "sad": "sad", "ps": "surprise", "surprise": "surprise",
    "neutral": None,  # no strong emotion -> all-zero label vector
}

def parse_ds2_audio_label(filename):
    stem = os.path.splitext(filename)[0].lower()
    token = stem.split("_")[-1]
    mapped = EMOTION_MAP_DS2_AUDIO.get(token)
    vec = np.zeros(NUM_CLASSES, dtype=np.float32)
    if mapped is not None:
        vec[EMOTION_LABELS.index(mapped)] = 1.0
    return vec


def load_ds2_audio():
    root = os.path.join(DATASET2_DIR, "Audio_Dataset", "Audio_Dataset")
    rows = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.lower().endswith(".wav"):
                path = os.path.join(dirpath, f)
                label = parse_ds2_audio_label(f)
                rows.append({"audio_path": path, "label": label})
    df = pd.DataFrame(rows)
    print(f"[Dataset2 Audio] found {len(df)} .wav files under {root}")
    return df


# ---- Dataset 2 video: RAVDESS encodes emotion as the 3rd '-'-separated field ----
RAVDESS_EMOTION_CODE_MAP = {
    "01": None, "02": None, "03": "happy", "04": "sad",
    "05": "angry", "06": "fear", "07": "disgust", "08": "surprise",
}

def parse_ravdess_label(filename):
    parts = os.path.splitext(filename)[0].split("-")
    vec = np.zeros(NUM_CLASSES, dtype=np.float32)
    if len(parts) >= 3:
        mapped = RAVDESS_EMOTION_CODE_MAP.get(parts[2])
        if mapped:
            vec[EMOTION_LABELS.index(mapped)] = 1.0
    return vec


def load_ds2_video():
    root = os.path.join(DATASET2_DIR, "Video_Dataset", "Video_Dataset")
    rows = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.lower().endswith(".mp4"):
                path = os.path.join(dirpath, f)
                label = parse_ravdess_label(f)
                rows.append({"video_path": path, "label": label})
    df = pd.DataFrame(rows)
    print(f"[Dataset2 Video] found {len(df)} .mp4 files under {root}")
    return df


ds2_audio_df = load_ds2_audio()
ds2_video_df = load_ds2_video()

# Diagnostic: how many samples landed on each emotion (helps catch a bad parse early)
def label_distribution(df, label_col):
    if len(df) == 0:
        return
    stacked = np.stack(df[label_col].values)
    for i, emo in enumerate(EMOTION_LABELS):
        print(f"  {emo}: {int(stacked[:, i].sum())}")
    print(f"  neutral/unlabeled (all-zero): {int((stacked.sum(axis=1) == 0).sum())}")

print("Dataset2 Audio label distribution:")
label_distribution(ds2_audio_df, "label")
print("Dataset2 Video label distribution:")
label_distribution(ds2_video_df, "label")


## 8. Train/val splits for Dataset 2, then combine with Dataset 1 for the Audio model

- **Text model** -> Dataset 1 (CMU-MOSEI) only, since Dataset 2 has no transcripts.
- **Audio model** -> Dataset 1 + Dataset 2 audio, combined.
- **Video model** -> Dataset 2 (RAVDESS) only, since Dataset 1 has no video.

In [ ]:
ds2_audio_train, ds2_audio_val = train_test_split(ds2_audio_df, test_size=0.15, random_state=42)
ds2_video_train, ds2_video_val = train_test_split(ds2_video_df, test_size=0.15, random_state=42)

print(f"Dataset2 audio: {len(ds2_audio_train)} train / {len(ds2_audio_val)} val")
print(f"Dataset2 video: {len(ds2_video_train)} train / {len(ds2_video_val)} val")

# ---- unified (path, label_vector) lists for the audio model ----
def cmu_rows_to_pairs(df):
    return [(row["audio_path"], np.array([row[e] for e in EMOTION_LABELS], dtype=np.float32))
            for _, row in df.iterrows()]

def ds2_rows_to_pairs(df, path_col):
    return [(row[path_col], row["label"]) for _, row in df.iterrows()]

audio_train_pairs = cmu_rows_to_pairs(cmu_train_df) + ds2_rows_to_pairs(ds2_audio_train, "audio_path")
audio_val_pairs = cmu_rows_to_pairs(cmu_val_df) + ds2_rows_to_pairs(ds2_audio_val, "audio_path")

print(f"Combined AUDIO model: {len(audio_train_pairs)} train / {len(audio_val_pairs)} val samples")

video_train_pairs = ds2_rows_to_pairs(ds2_video_train, "video_path")
video_val_pairs = ds2_rows_to_pairs(ds2_video_val, "video_path")

print(f"VIDEO model: {len(video_train_pairs)} train / {len(video_val_pairs)} val samples")


## 9. tf.data pipelines

In [ ]:
def make_audio_tf_dataset(pairs, batch_size=BATCH_SIZE, shuffle=True):
    paths = [p for p, _ in pairs]
    labels = np.stack([l for _, l in pairs])
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), reshuffle_each_iteration=True)

    def _load(path, label):
        feat = tf.py_function(func=extract_audio_features, inp=[path], Tout=tf.float32)
        feat.set_shape([AUDIO_TIME_STEPS, AUDIO_FEATURE_DIM])
        label.set_shape([NUM_CLASSES])
        return feat, label

    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def make_video_tf_dataset(pairs, batch_size=BATCH_SIZE, shuffle=True):
    paths = [p for p, _ in pairs]
    labels = np.stack([l for _, l in pairs])
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), reshuffle_each_iteration=True)

    def _load(path, label):
        frames = tf.py_function(func=extract_face_sequence, inp=[path], Tout=tf.float32)
        frames.set_shape([VIDEO_NUM_FRAMES, VIDEO_FRAME_SIZE, VIDEO_FRAME_SIZE, 3])
        label.set_shape([NUM_CLASSES])
        return frames, label

    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def make_text_tf_dataset(df, vectorizer, batch_size=BATCH_SIZE, shuffle=True):
    texts = df["text"].astype(str).tolist()
    labels = df[EMOTION_LABELS].values.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((texts, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(texts), reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    ds = ds.map(lambda x, y: (vectorizer(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

print("tf.data pipeline builders ready.")


## 10. MODEL 1 — Text (Embedding + BiLSTM + ANN)

In [ ]:
def build_text_model():
    inputs = layers.Input(shape=(TEXT_MAX_LEN,), dtype="int32", name="text_tokens")
    x = layers.Embedding(input_dim=TEXT_VOCAB_SIZE, output_dim=TEXT_EMBED_DIM, mask_zero=True)(inputs)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
    x = layers.Bidirectional(layers.LSTM(64))(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    embedding = layers.Dense(128, activation="relu", name="text_emotion_embedding")(x)
    x = layers.Dropout(0.3)(embedding)
    outputs = layers.Dense(NUM_CLASSES, activation="sigmoid", name="text_output")(x)

    model = Model(inputs, outputs, name="TextEmotionModel")
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
                  loss="binary_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return model


text_vectorizer = layers.TextVectorization(max_tokens=TEXT_VOCAB_SIZE, output_mode="int",
                                            output_sequence_length=TEXT_MAX_LEN)
text_vectorizer.adapt(cmu_train_df["text"].astype(str).tolist())

text_train_ds = make_text_tf_dataset(cmu_train_df, text_vectorizer, shuffle=True)
text_val_ds = make_text_tf_dataset(cmu_val_df, text_vectorizer, shuffle=False)

text_model = build_text_model()
text_model.summary()


In [ ]:
text_callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
]

text_history = text_model.fit(
    text_train_ds, validation_data=text_val_ds,
    epochs=EPOCHS_TEXT, callbacks=text_callbacks,
)


In [ ]:
text_model_path = os.path.join(SAVE_DIR, "text_model_final.h5")
text_model.save(text_model_path)

vocab_path = os.path.join(SAVE_DIR, "text_vectorizer_vocab.pkl")
pickle.dump(text_vectorizer.get_vocabulary(), open(vocab_path, "wb"))

print("Saved:", text_model_path)
print("Saved:", vocab_path)


## 11. MODEL 2 — Audio (1D-CNN + LSTM), trained on Dataset 1 + Dataset 2 combined

In [ ]:
def build_audio_model():
    inputs = layers.Input(shape=(AUDIO_TIME_STEPS, AUDIO_FEATURE_DIM), name="audio_features")

    x = layers.Conv1D(64, 5, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Conv1D(128, 5, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Conv1D(256, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.LSTM(128, return_sequences=True)(x)
    x = layers.LSTM(64)(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    embedding = layers.Dense(128, activation="relu", name="speech_emotion_embedding")(x)
    x = layers.Dropout(0.3)(embedding)
    outputs = layers.Dense(NUM_CLASSES, activation="sigmoid", name="audio_output")(x)

    model = Model(inputs, outputs, name="SpeechEmotionModel")
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
                  loss="binary_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return model


audio_train_ds = make_audio_tf_dataset(audio_train_pairs, shuffle=True)
audio_val_ds = make_audio_tf_dataset(audio_val_pairs, shuffle=False)

audio_model = build_audio_model()
audio_model.summary()


In [ ]:
audio_callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
]

audio_history = audio_model.fit(
    audio_train_ds, validation_data=audio_val_ds,
    epochs=EPOCHS_AUDIO, callbacks=audio_callbacks,
)


In [ ]:
audio_model_path = os.path.join(SAVE_DIR, "audio_model_final.h5")
audio_model.save(audio_model_path)
print("Saved:", audio_model_path)


## 12. MODEL 3 — Video (TimeDistributed CNN + LSTM), trained on Dataset 2 (RAVDESS) only

In [ ]:
def build_video_model():
    frame_input = layers.Input(shape=(VIDEO_FRAME_SIZE, VIDEO_FRAME_SIZE, 3))
    c = layers.Conv2D(32, 3, activation="relu", padding="same")(frame_input)
    c = layers.BatchNormalization()(c)
    c = layers.MaxPooling2D()(c)
    c = layers.Conv2D(64, 3, activation="relu", padding="same")(c)
    c = layers.BatchNormalization()(c)
    c = layers.MaxPooling2D()(c)
    c = layers.Conv2D(128, 3, activation="relu", padding="same")(c)
    c = layers.BatchNormalization()(c)
    c = layers.GlobalAveragePooling2D()(c)
    cnn_backbone = Model(frame_input, c, name="CustomCNNBackbone")

    inputs = layers.Input(shape=(VIDEO_NUM_FRAMES, VIDEO_FRAME_SIZE, VIDEO_FRAME_SIZE, 3), name="video_frames")
    x = layers.TimeDistributed(cnn_backbone)(inputs)
    x = layers.LSTM(128, return_sequences=True)(x)
    x = layers.LSTM(64)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    embedding = layers.Dense(128, activation="relu", name="visual_emotion_embedding")(x)
    x = layers.Dropout(0.3)(embedding)
    outputs = layers.Dense(NUM_CLASSES, activation="sigmoid", name="video_output")(x)

    model = Model(inputs, outputs, name="FacialEmotionModel")
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
                  loss="binary_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return model


video_train_ds = make_video_tf_dataset(video_train_pairs, shuffle=True)
video_val_ds = make_video_tf_dataset(video_val_pairs, shuffle=False)

video_model = build_video_model()
video_model.summary()


In [ ]:
video_callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
]

video_history = video_model.fit(
    video_train_ds, validation_data=video_val_ds,
    epochs=EPOCHS_VIDEO, callbacks=video_callbacks,
)


In [ ]:
video_model_path = os.path.join(SAVE_DIR, "video_model_final.h5")
video_model.save(video_model_path)
print("Saved:", video_model_path)


## 13. Confirm everything landed on Drive

In [ ]:
!ls -lh "{SAVE_DIR}"


## 14. Quick sanity-check inference on the freshly trained models

In [ ]:
# Text
sample_text = ["I've been feeling really overwhelmed and tired lately"]
tokens = text_vectorizer(sample_text)
print("Text prediction:", dict(zip(EMOTION_LABELS, text_model.predict(tokens, verbose=0)[0])))

# Audio (uses the first available training file as a smoke test)
if len(audio_train_pairs) > 0:
    sample_path = audio_train_pairs[0][0]
    feat = np.expand_dims(extract_audio_features(sample_path), 0)
    print("Audio prediction:", dict(zip(EMOTION_LABELS, audio_model.predict(feat, verbose=0)[0])))

# Video
if len(video_train_pairs) > 0:
    sample_path = video_train_pairs[0][0]
    frames = np.expand_dims(extract_face_sequence(sample_path), 0)
    print("Video prediction:", dict(zip(EMOTION_LABELS, video_model.predict(frames, verbose=0)[0])))


## Next step

Download `text_model_final.h5`, `audio_model_final.h5`, `video_model_final.h5`, and
`text_vectorizer_vocab.pkl` from `SAVE_DIR` on your Drive, and drop them into the
`models/` folder of the FastAPI + Streamlit app from earlier — no code changes
needed there as long as `EMOTION_LABELS` in `backend/model_paths.py` still reads
`["happy", "sad", "angry", "surprise", "disgust", "fear"]`, which matches what
these 3 models were trained on.